# 1. From deformation to energy and stress

**Goal:** use Torch to compute a local material density, stress, and stress derivatives.

A body has many material points. At each point, the $3\times3$ deformation gradient
$F$ describes how small vectors change. The plastic deformation gradient $F_p$
describes the locally relaxed state. We use the split $F=F_e F_p$, so
$F_e=F F_p^{-1}$ measures deformation relative to that state. When $F_p=I$,
$F=\operatorname{diag}(s,1,1)$ stretches the first direction by $s$.

We use dimensionless geometry and consistent arbitrary units throughout this series.
Density $W$ is energy per original reference volume. Total energy $E$ integrates
that density over the body. Run every cell in order using the project's Python
environment. This notebook runs independently of the other lessons.

In [ ]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline

torch.set_num_threads(1)
dtype = torch.float64

## Write the constitutive law

Use a compressible Neo-Hookean density with parameters $\theta=(\lambda,\mu)$.
First compute the elastic energy per unit relaxed volume,

$$\psi(F_e,\theta)=\frac{\mu}{2}(\|F_e\|^2-3)-\mu\log J_e+
\frac{\lambda}{2}(\log J_e)^2,\qquad J_e=\det F_e>0.$$

A unit of original reference volume becomes $\det F_p$ units of relaxed volume.
The density per original reference volume is therefore

$$W(F,F_p,\theta)=\det F_p\,\psi(F F_p^{-1},\theta).$$

The inputs are `F: (N, 3, 3)`, `Fp: (N, 3, 3)`, and `theta: (N, 2)`.
Each entry of `W: (N,)` depends only on the corresponding row of these inputs.
We use states with $\det F_p>0$, $J_e>0$, $\lambda\ge 0$, and $\mu>0$.

In [ ]:
def density(F, Fp, theta):
    Fe = F @ torch.linalg.inv(Fp)
    lam, mu = theta.unbind(-1)
    log_J = torch.linalg.det(Fe).log()
    psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
    return torch.linalg.det(Fp) * psi

I = torch.eye(3, dtype=dtype)
F = torch.tensor([
    [[1.1, 0., 0.], [0., 1., 0.], [0., 0., 1.]],
    [[1.15, .12, 0.], [.02, .96, .06], [0., .03, 1.08]],
], dtype=dtype, requires_grad=True)
Fp = torch.stack((I, torch.tensor([
    [1.04, .08, 0.], [0., .98, .05], [0., 0., 1.02],
], dtype=dtype))).requires_grad_()
theta = torch.tensor([[2., 3.], [1.5, 2.5]], dtype=dtype, requires_grad=True)

W = density(F, Fp, theta)
P = torch.autograd.grad(W.sum(), F, create_graph=True)[0]
print('density per point:', W.detach())
print('first Piola stress per point:\n', P.detach())

## Why differentiate a sum?

First Piola stress is $P_{ij}=\partial W/\partial F_{ij}$, with $F_p$ and $\theta$
held fixed. We pass `W.sum()` to `autograd.grad` so that it has one scalar output
to differentiate with respect to the whole input `F`.

Think of two independent points with inputs `x = [2, 3]` and outputs
`x**2 = [4, 9]`. Their total is 13. If you nudge `x[0]` to 2.001, the total
becomes $4.004001+9$. The second term stays at 9. Dividing the change by
0.001 gives 4.001, close to the derivative 4. The derivative with respect
to `x[1]` is 6, so the gradient of the total with respect to `x` is `[4, 6]`.

The same reasoning applies to `W.sum()`. Changing `F[n]` only changes `W[n]`,
so the gradient at `F[n]` is that point's own stress. The sum combines
material points for one backward pass. A mesh energy uses reference-volume weights.

`torch.autograd.grad` returns a tuple with one gradient tensor per input. Here
`F` is the only input, so `[0]` in `grad(W.sum(), F, ...)[0]` extracts the whole
`(N, 3, 3)` gradient tensor. Index the result as `P[0]` to select the first point.

Check the first point's `P[0, 0, 0]` by perturbing `F[0, 0, 0]`. We also check
that density and stress vanish when $F=F_p$, since then $F_e=I$.

In [ ]:
h = 1e-6
f, fp, params = F.detach(), Fp.detach(), theta.detach()
step = torch.zeros_like(f)
step[0, 0, 0] = h
fd = (density(f + step, fp, params) - density(f - step, fp, params)) / (2 * h)
torch.testing.assert_close(P[0, 0, 0], fd[0], rtol=1e-6, atol=1e-9)

relaxed = fp.clone().requires_grad_()
relaxed_W = density(relaxed, fp, params)
relaxed_P = torch.autograd.grad(relaxed_W.sum(), relaxed)[0]
assert relaxed_W.abs().max() < 1e-12 and relaxed_P.abs().max() < 1e-12
print('stress finite-difference error:', (P[0, 0, 0] - fd[0]).abs().item())

## Differentiate the stress

### Keep the first derivative differentiable

The first backward pass computed $P=\partial W/\partial F$. With
`create_graph=True`, PyTorch also records the operations that compute this
derivative. That lets another backward pass follow how $P$ depends on the
inputs. For a scalar example, take $W=x^3$.

```python
x = torch.tensor(3.0, requires_grad=True)
p = torch.autograd.grad(x**3, x, create_graph=True)[0]  # 3*x**2 = 27
dp_dx = torch.autograd.grad(p, x)[0]                  # 6*x = 18
```

With the default `create_graph=False`, the first derivative in this example
still has value 27. The second call would fail because PyTorch did not record
the operations that produced `p`.

### Why nine more backward passes?

At each material point, each of the nine stress components gets its own
Jacobian row. `P.flatten(1)` reshapes `(N, 3, 3)` to `(N, 9)`.
`unbind(1)` takes one column at a time, starting with `P[:, 0, 0]`, then
`P[:, 0, 1]`, and ending with `P[:, 2, 2]`. Each column has shape `(N,)`.

For example, with two points the second call differentiates
`P[0, 0, 1] + P[1, 0, 1]`. Changing `Fp[0]` only changes the first term, and
changing `Fp[1]` only changes the second. So one call recovers the row for
$P_{01}$ at **every point**. Each call returns three gradients in input order
`(F, Fp, theta)`. Summing all nine stress components would add the nine
Jacobian rows together. The loop keeps them separate by selecting one stress
component per call.

`create_graph=True` made the first derivative differentiable.
`retain_graph=True` keeps that graph available for the next stress component
after each call.

### Assemble the Jacobians

`stress_rows` is grouped by stress component. `zip(*stress_rows)` takes the
first gradient from every tuple to make the `F` group, the second to make the
`Fp` group, and the third to make the `theta` group.

For a mental picture, shrink the stress matrix to $2\times2$. Its four output
components give four gradient tables `A, B, C, D` in that order. `stack`
gives them a single index. `unflatten` puts them back in the same row and
column positions as the output components.

```text
Before unflatten           After unflatten
q = 0, 1, 2, 3             (0, 0)  (0, 1)     A  B
    A  B  C  D             (1, 0)  (1, 1)     C  D
```

In the real `Fp` calculation, each of the nine gradient tables has shape
`(N, 3, 3)`. `stack(dim=1)` inserts the table index after the batch axis to
get `(N, 9, 3, 3)`. `unflatten(1, (3, 3))` splits that index back into the
stress indices `i, j`. The input indices `k, l` stay at the end.

```python
_, fp_rows, _ = zip(*stress_rows)
stacked = torch.stack(fp_rows, dim=1)       # (N, 9, 3, 3)
dPdFp = stacked.unflatten(1, (3, 3))       # (N, 3, 3, 3, 3)
```

For example, `stacked[n, 1, 1, 2]` becomes `dPdFp[n, 0, 1, 1, 2]` because
table `q = 3*0 + 1` corresponds to `P[n, 0, 1]`. Both entries answer the
same question. If we increase `Fp[n, 1, 2]` by a small amount, how much
does `P[n, 0, 1]` change per unit increase?

| Tensor | Shape | Entry for material point $n$ |
| --- | --- | --- |
| `dPdF` | `(N, 3, 3, 3, 3)` | $\partial P_{ij}/\partial F_{kl}$ |
| `dPdFp` | `(N, 3, 3, 3, 3)` | $\partial P_{ij}/\partial (F_p)_{kl}$ |
| `dPdtheta` | `(N, 3, 3, 2)` | $\partial P_{ij}/\partial \theta_a$, with $a=0$ for $\lambda$ and $a=1$ for $\mu$ |

Each derivative varies one input while holding the other two fixed. Varying
$F_p$ changes both $F_e$ and the volume factor $\det F_p$. A plastic evolution
law would describe how $F_p$ itself changes during loading.

In [ ]:
stress_rows = [
    torch.autograd.grad(component.sum(), (F, Fp, theta), retain_graph=True)
    for component in P.flatten(1).unbind(1)
]
dPdF, dPdFp, dPdtheta = [
    torch.stack(rows, dim=1).unflatten(1, (3, 3))
    for rows in zip(*stress_rows)
]
print('dPdF:', dPdF.shape)
print('dPdFp:', dPdFp.shape)
print('dPdtheta:', dPdtheta.shape)

## Check the stress derivatives

Pick one input component, for example $(F_p)_{12}$. Increase it by $h$, decrease
it by $h$, and recompute the full stress matrix each time. The central difference
$(P_+-P_-)/(2h)$ estimates how **all nine** stress components respond to that
one input change. Those nine values form a Jacobian column. The loop compares
each such column against autograd.

We apply the same perturbation to every sample in parallel because the points
are independent. The second sample has nonidentity $F_p$ and shear, so the
check exercises the plastic volume factor and off-diagonal entries.

In [ ]:
def stress_at(F, Fp, theta):
    F = F.detach().requires_grad_()
    return torch.autograd.grad(density(F, Fp, theta).sum(), F)[0]

inputs = (f, fp, params)
for arg, (name, jacobian) in enumerate((
    ('dPdF', dPdF), ('dPdFp', dPdFp), ('dPdtheta', dPdtheta),
)):
    columns = jacobian.flatten(3)
    errors = []
    for column in range(columns.shape[-1]):
        step = torch.zeros_like(inputs[arg])
        step.flatten(1)[:, column] = h
        plus, minus = list(inputs), list(inputs)
        plus[arg] = inputs[arg] + step
        minus[arg] = inputs[arg] - step
        fd = (stress_at(*plus) - stress_at(*minus)) / (2 * h)
        actual = columns[..., column]
        torch.testing.assert_close(actual, fd, rtol=1e-6, atol=1e-8)
        errors.append((actual - fd).abs().max().item())
    print(f'{name} max finite-difference error: {max(errors):.2e}')

### Visualize the Jacobians

Plot the second material point (`n = 1`), which has shear and nonidentity $F_p$.
Each row selects the stress component to observe. Each column selects the input
component to perturb while holding the other components fixed. For example, in
the middle panel, row $P_{01}$ and column $(F_p)_{12}$ show `dPdFp[1, 0, 1, 1, 2]`.

Red means a small positive perturbation increases that stress component. Blue
means it decreases. White is near zero. Read magnitudes against each panel's
own color scale. All component indices are zero-based.

In [ ]:
n = 1
indices = [f'{i}{j}' for i in range(3) for j in range(3)]
stress_labels = [f'$P_{{{ij}}}$' for ij in indices]
panels = (
    (dPdF, r'$\partial P/\partial F$', [f'$F_{{{ij}}}$' for ij in indices]),
    (dPdFp, r'$\partial P/\partial F_p$', [f'$F_{{p,{ij}}}$' for ij in indices]),
    (dPdtheta, r'$\partial P/\partial \theta$', [r'$\lambda$', r'$\mu$']),
)
fig, axes = plt.subplots(1, 3, figsize=(12, 4), width_ratios=(1, 1, .55),
                         sharey=True, constrained_layout=True)
for ax, (jacobian, title, input_labels) in zip(axes, panels):
    matrix = jacobian[n].reshape(9, -1).detach()
    limit = matrix.abs().max().item() or 1.0
    image = ax.imshow(matrix.numpy(), cmap='RdBu_r', vmin=-limit, vmax=limit,
                      aspect='auto', interpolation='nearest')
    ax.set(title=title, xlabel='Input component')
    ax.set_xticks(range(len(input_labels)), input_labels)
    fig.colorbar(image, ax=ax)
axes[0].set_yticks(range(9), stress_labels)
axes[0].set_ylabel('Stress component')
fig.suptitle(f'Stress Jacobians at material point {n}')
plt.show()

#### Read the colors

Compare the $P_{00}$ row in the first two panels. The red cell in the $F_{00}$
column says that increasing the total stretch increases $P_{00}$. The blue
cell in the $F_{p,00}$ column says that increasing the relaxed stretch at fixed
total stretch decreases $P_{00}$. In a purely axial stretch, the elastic stretch
is $s/s_p$. Increasing the numerator increases the elastic stretch, while
increasing the denominator decreases it.

The matrix version near the identity is $F_e\approx I+H-H_p$, where
$H=F-I$ and $H_p=F_p-I$. The same component of $F$ and $F_p$ therefore moves
the elastic deformation in opposite directions to first order. At the identity,
the Neo-Hookean stiffness for a same-component perturbation is $\lambda+2\mu$
for a normal component and $\mu$ for a shear component. Both are positive.
There $\partial P/\partial F_p=-\partial P/\partial F$, which explains the
mostly red and mostly blue diagonals near that state. In the plotted state,
shear and the dependence of the stress on
$F_p$ through the volume factor and the elastic-to-total stress transformation
also affect the colors.

In the right panel, the deformation stays fixed. Both parameter columns are
red in the $P_{00}$ row, so increasing either parameter increases that stress
component at this state. In the $P_{11}$ row the colors have opposite signs.
Use each panel's color bar when comparing magnitudes across panels.

## Observe a whole loading path

Evaluate many independent stretches with $F=\operatorname{diag}(s,1,1)$,
$F_p=I$, and fixed $\theta$. The identity state has zero energy and stress.
Compression and extension both cost energy. Along this path only the first
diagonal entry of $F$ changes, so the slope of the energy curve with respect
to $s$ is the axial stress plotted on the right. The plot calls it $P_{11}$
using one-based component labels, and the code reads it as `[:, 0, 0]`.

In [ ]:
stretch = torch.linspace(0.8, 1.2, 81, dtype=dtype)
path = I.repeat(len(stretch), 1, 1)
path[:, 0, 0] = stretch
path.requires_grad_()
values = density(path, I.expand_as(path), params[:1].expand(len(stretch), -1))
stress = torch.autograd.grad(values.sum(), path)[0][:, 0, 0]
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
axes[0].plot(stretch.numpy(), values.detach().numpy())
axes[1].plot(stretch.numpy(), stress.detach().numpy())
for ax, label in zip(axes, ['Energy density W', 'First Piola stress P11']):
    ax.set(xlabel='Stretch s', ylabel=label)
    ax.axvline(1, color='gray', linestyle=':')
    ax.grid(alpha=0.25)
plt.show()

## Try it

Plot each modified loading path against the original on the same axes.

1. Keep $F_p=I$ and double both $\lambda$ and $\mu$. How do the energy and
   stress change at each stretch? Does the zero move?
2. Restore the original material parameters and set
   $F_p=\operatorname{diag}(1.05,1,1)$. Where are the energy minimum and
   the stress zero now?

1. Energy and stress both double at each stretch because $W$ is linear in
   the two parameters. The zero stays at $s=1$.
2. The energy minimum and stress zero move to $s=1.05$. At that stretch,
   $F=F_p$, so the elastic stretch $s/1.05$ is 1 and $F_e=I$.

All three cases share the same axes. The vertical guides mark the two relaxed stretches.

In [ ]:
path_params = params[:1].expand(len(stretch), -1)
doubled_W = density(path, I.expand_as(path), 2 * path_params)
doubled_stress = torch.autograd.grad(doubled_W.sum(), path)[0][:, 0, 0]

shifted_Fp = torch.diag(path.new_tensor([1.05, 1., 1.]))
shifted_W = density(path, shifted_Fp.expand_as(path), path_params)
shifted_stress = torch.autograd.grad(shifted_W.sum(), path)[0][:, 0, 0]

torch.testing.assert_close(doubled_W, 2 * values)
torch.testing.assert_close(doubled_stress, 2 * stress)
relaxed_stretch = shifted_Fp[0, 0]
relaxed_index = (stretch - relaxed_stretch).abs().argmin()
torch.testing.assert_close(stretch[relaxed_index], relaxed_stretch)
assert shifted_W.argmin() == relaxed_index
assert shifted_W[relaxed_index].abs() < 1e-12 and shifted_stress[relaxed_index].abs() < 1e-12

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for energy, axial_stress, label, style in (
    (values, stress, 'Original', '-'),
    (doubled_W, doubled_stress, r'Double $\lambda$ and $\mu$', '--'),
    (shifted_W, shifted_stress, r'$F_p = \mathrm{diag}(1.05, 1, 1)$', '-.'),
):
    axes[0].plot(stretch, energy.detach(), style, label=label)
    axes[1].plot(stretch, axial_stress.detach(), style, label=label)
for ax, label in zip(axes, ['Energy density W', 'First Piola stress P11']):
    ax.set(xlabel='Stretch s', ylabel=label)
    ax.axvline(1, color='gray', linestyle=':', linewidth=1)
    ax.axvline(relaxed_stretch.item(), color='gray', linestyle=':', linewidth=1)
    ax.grid(alpha=0.25)
axes[0].legend()
plt.show()

**Checkpoint** You can compute a row-independent density, its first Piola stress,
and the partial derivatives of that stress with respect to all three inputs.
Next, put the same law into native FEM in lesson 2.